## 03 Vegetation Condition: MODIS NDVI on Tribal Lands
**Series:** Tribal Agriculture & Land Health in South Dakota  
**Author:** Lilly Jones, PhD  
**Primary Focus:** Oglala Lakota (Pine Ridge), Sicangu Lakota (Rosebud)  
**In Scope:** All South Dakota Tribal Nations  
**Data Sources:** MODIS MOD13Q1 (NDVI) via ORNL DAAC MODIS Web Service

## Purpose
The Normalized Difference Vegetation Index (NDVI) is the closest publicly
available proxy for what Tribal land managers measure directly as pasture
condition score. Both reflect whether the vegetation on a given piece of land
is green, dense, and growing or dry, sparse, and stressed.

This notebook uses MODIS MOD13Q1 16-day composite NDVI (250m resolution)
retrieved via the ORNL DAAC MODIS Web Service API. Point-based time series
are extracted for each Tribal land centroid, then aggregated to growing-season
annual means. No raster downloads or local files are required.

## What NDVI Can and Cannot See
NDVI measures vegetation greenness from satellite. It is a useful regional
proxy, but it cannot see:
- Species composition (cheatgrass looks green just like native grasses)
- Soil compaction or root health below the surface
- Palatability or forage quality
- Cultural significance of specific plant communities
- Pasture-level management history

This is precisely why Tribal-led pasture condition scoring (the operational
pipeline) matters. NDVI shows the pattern from above; pasture condition
scores show the reality on the ground.

## Data Approach
The ORNL DAAC MODIS Web Service provides time series for point locations
without requiring raster downloads or NASA Earthdata accounts. Queries return
16-day NDVI values for up to a 7×7 pixel subset (1.75 km × 1.75 km) centered
on each point. We use Tribal land centroids and average the pixel subset.

## Research Questions
- What is the growing-season NDVI trend on Pine Ridge and Rosebud, 2000–present?
- Which years correspond to the severe drought years identified in notebook 02?
- Are there multi-year downward trends that suggest cumulative pasture stress?
- How does NDVI variability compare across SD Tribal Nations?

In [ ]:
# Imports
import sys
from pathlib import Path

REPO_ROOT = Path().resolve().parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import json
import time
import warnings
from datetime import datetime

import contextily as ctx
import geopandas as gpd
gpd.options.io_engine = "fiona"
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import pandas as pd
import requests
import seaborn as sns
from scipy import stats
from shapely.validation import make_valid
from tenacity import retry, stop_after_attempt, wait_exponential

from src.data import constants
from src.data.constants import (
    SD_TRIBES_ALL, SD_TRIBES_PRIMARY,
    CENSUS_NAME_MAP, CENSUS_TO_COMMON,
    NDVI_THRESHOLDS,
    CRS_GEOGRAPHIC, CRS_PROJECTED,
)
from src.indigenous.sovereignty import print_data_acknowledgment, generate_citations

warnings.filterwarnings("ignore", category=FutureWarning)
%matplotlib inline

print(f"Repo root : {REPO_ROOT}")
print(f"Analysis run: {datetime.now().strftime('%Y-%m-%d %H:%M')}")

In [ ]:
# Print data acknowledgement at the top of every notebook
print_data_acknowledgment(source_keys=["census_aiannh", "modis_ndvi"])

## Configure

In [ ]:
# Analysis parameters
# MODIS MOD13Q1 coverage begins 2000-02-18
NDVI_START_YEAR = 2000
NDVI_END_YEAR   = 2024

# Growing season for mixed-grass prairie (May–September)
GROWING_SEASON_MONTHS = [5, 6, 7, 8, 9]

# MOD13Q1 scale factor (raw values are integer × 0.0001)
NDVI_SCALE = 0.0001

# NDVI fill/invalid value in MOD13Q1
NDVI_FILL_VALUE = -3000

# ORNL DAAC MODIS Web Service: no account required for point queries
ORNL_BASE     = "https://modis.ornl.gov/rst/api/v1"
MODIS_PRODUCT = "MOD13Q1"     # Terra 16-day 250m NDVI
MODIS_BAND    = "_250m_16_days_NDVI"

# Pixel subset size for averaging (km from center, max 5.5km)
# 2km gives a ~4×4 pixel average at 250m, appropriate for pasture-scale analysis
SUBSET_KM = 2

# NDVI condition thresholds (from config)
NDVI_POOR      = NDVI_THRESHOLDS["poor"]      # 0.25
NDVI_FAIR      = NDVI_THRESHOLDS["fair"]      # 0.35
NDVI_GOOD      = NDVI_THRESHOLDS["good"]      # 0.45
NDVI_EXCELLENT = NDVI_THRESHOLDS["excellent"]  # 0.55

print("NDVI ANALYSIS CONFIGURATION")
print(f"  Product          : {MODIS_PRODUCT} (16-day, 250m)")
print(f"  Band             : {MODIS_BAND}")
print(f"  Period           : {NDVI_START_YEAR}–{NDVI_END_YEAR}")
print(f"  Growing season   : months {GROWING_SEASON_MONTHS}")
print(f"  Pixel subset     : ±{SUBSET_KM} km")
print(f"\nNDVI condition thresholds (growing season mean):")
print(f"  Poor     : < {NDVI_POOR}")
print(f"  Fair     : {NDVI_POOR}–{NDVI_FAIR}")
print(f"  Good     : {NDVI_FAIR}–{NDVI_GOOD}")
print(f"  Excellent: > {NDVI_GOOD}")

## Load Tribal Boundaries and Centroids

In [ ]:
# Tribal boundaries
GEOJSON_PATH = constants.OUTPUTS_DIR/"sd_tribal_land_base.geojson"
CACHE_PATH   = constants.CACHE_DIR/"tl_2023_us_aiannh.geojson"

if GEOJSON_PATH.exists():
    tribal_lands = gpd.read_file(GEOJSON_PATH)
    print(f"Loaded from notebook 01 output: {len(tribal_lands)} Tribal Nations")
elif CACHE_PATH.exists():
    all_aiannh = gpd.read_file(CACHE_PATH)
    census_names = list(CENSUS_NAME_MAP.values())
    tribal_lands = all_aiannh[all_aiannh["NAME"].isin(census_names)].copy()
    tribal_lands = tribal_lands.dissolve(by="NAME", as_index=False)
    tribal_lands["geometry"]    = tribal_lands.geometry.apply(make_valid)
    tribal_lands["common_name"] = tribal_lands["NAME"].map(CENSUS_TO_COMMON)
    tribal_lands["area_km2"]    = tribal_lands.to_crs(CRS_PROJECTED).geometry.area / 1e6
    tribal_lands["is_primary"]  = tribal_lands["common_name"].isin(SD_TRIBES_PRIMARY)
    print(f"Loaded from AIANNH cache: {len(tribal_lands)} Tribal Nations")
else:
    raise FileNotFoundError("Run notebook 01 first to create sd_tribal_land_base.geojson")

# Projected centroids → geographic for API queries
tribal_proj    = tribal_lands.to_crs(CRS_PROJECTED)
centroids_proj = tribal_proj.geometry.centroid
centroids_geo  = centroids_proj.to_crs(CRS_GEOGRAPHIC)
tribal_lands["centroid_lat"] = centroids_geo.y
tribal_lands["centroid_lon"] = centroids_geo.x

print(tribal_lands[["common_name", "centroid_lat", "centroid_lon", "is_primary"]].to_string(index=False))

## Fetch MODIS NDVI via ORNL DAAC Web Service
The ORNL DAAC MODIS Web Service returns 16-day NDVI time series for a
point location as JSON. No NASA Earthdata account or local raster files
required. Results are cached to `data/cache/`.
**First run:** ~5–10 minutes for SD Tribal Nations × 24 years of data.  
**Subsequent runs:** instant from cache.

In [ ]:
# ORNL DAAC MODIS Web Service fetch
_retry = retry(
    stop=stop_after_attempt(3),
    wait=wait_exponential(multiplier=1, min=2, max=10),
    reraise=True,
)

@_retry
def fetch_modis_ndvi(
    lat: float,
    lon: float,
    start_date: str,
    end_date: str,
    km_above_below: int = SUBSET_KM,
    km_left_right: int  = SUBSET_KM,
) -> pd.DataFrame:
    """
    Fetch MODIS MOD13Q1 NDVI time series for a point location via ORNL DAAC.

    Parameters
    lat, lon       : WGS84 coordinates
    start_date/end_date: ISO dates; converted to ORNL's required AYYYYDDD format
    km_above_below : km above/below center pixel to average (default 2)
    km_left_right  : km left/right of center pixel to average (default 2)

    Returns
    DataFrame with columns: date, ndvi (scaled 0–1), pixel_count
    """
    start_modis = pd.Timestamp(start_date).strftime("A%Y%j")
    end_modis = pd.Timestamp(end_date).strftime("A%Y%j")
    # ORNL permits at most 10 16-day composite tiles per request.
    windows = []
    cursor = pd.Timestamp(start_date)
    end = pd.Timestamp(end_date)
    while cursor <= end:
        window_end = min(cursor + pd.Timedelta(days=150), end)
        windows.append((cursor.strftime("A%Y%j"), window_end.strftime("A%Y%j")))
        cursor = window_end + pd.Timedelta(days=1)
    subsets = []
    headers = {"Accept": "application/json"}
    for window_start, window_end in windows:
        url = (f"{ORNL_BASE}/{MODIS_PRODUCT}/subset?latitude={lat}&longitude={lon}"
               f"&startDate={window_start}&endDate={window_end}"
               f"&kmAboveBelow={km_above_below}&kmLeftRight={km_left_right}")
        r = requests.get(url, headers=headers, timeout=120)
        r.raise_for_status()
        subsets.extend(r.json().get("subset", []))
        time.sleep(0.1)

    records = []
    for subset in subsets:
        if subset.get("band") != MODIS_BAND:
            continue
        raw_vals = [
            v for v in subset.get("data", [])
            if v is not None and v > NDVI_FILL_VALUE
        ]
        if not raw_vals:
            continue
        ndvi_mean = np.mean(raw_vals) * NDVI_SCALE
        # MOD13Q1 calendar date is in modis_date field or calendar_date
        cal_date = subset.get("calendar_date", subset.get("modis_date", ""))
        try:
            date = pd.to_datetime(cal_date)
        except Exception:
            continue
        records.append({
            "date":        date,
            "ndvi":        round(ndvi_mean, 4),
            "pixel_count": len(raw_vals),
        })

    return pd.DataFrame(records).drop_duplicates(subset=["date"]).sort_values("date").reset_index(drop=True)


print("ORNL DAAC MODIS Web Service loader defined.")
print(f"Product : {MODIS_PRODUCT}")
print(f"Band    : {MODIS_BAND}")
print(f"Endpoint: {ORNL_BASE}")

In [ ]:
# Download NDVI per Tribal land centroid
START_DATE = f"{NDVI_START_YEAR}-01-01"
END_DATE   = f"{NDVI_END_YEAR}-12-31"

try:
    constants.CACHE_DIR.mkdir(parents=True, exist_ok=True)
except FileExistsError:
    pass

ndvi_parts = []
failed     = []

for _, tribe in tribal_lands.iterrows():
    name = tribe["common_name"]
    lat  = tribe["centroid_lat"]
    lon  = tribe["centroid_lon"]

    cache_file = constants.CACHE_DIR/f"ndvi_{name.replace(' ', '_').lower()}.csv"

    if cache_file.exists():
        df = pd.read_csv(cache_file, parse_dates=["date"])
        print(f"  {name}: loaded from cache ({len(df)} observations)")
    else:
        try:
            print(f"  {name}: downloading...")
            df = fetch_modis_ndvi(
                lat=lat, lon=lon,
                start_date=START_DATE, end_date=END_DATE,
            )
            if df.empty:
                raise ValueError("Empty response: check coordinates")
            df.to_csv(cache_file, index=False)
            print(f"    → {len(df)} observations cached")
            time.sleep(0.5)  # be polite to ORNL servers
        except Exception as e:
            print(f"  {name}: FAILED {e}")
            failed.append({"name": name, "error": str(e)})
            continue

    df["common_name"] = name
    df["is_primary"]  = tribe["is_primary"]
    ndvi_parts.append(df)

if ndvi_parts:
    ndvi_raw = pd.concat(ndvi_parts, ignore_index=True)
    ndvi_raw["date"]  = pd.to_datetime(ndvi_raw["date"])
    ndvi_raw["year"]  = ndvi_raw["date"].dt.year
    ndvi_raw["month"] = ndvi_raw["date"].dt.month
    print(f"\nNDVI data loaded: {len(ndvi_raw):,} observations across {ndvi_raw['common_name'].nunique()} Tribal Nations")
else:
    raise RuntimeError("No NDVI data loaded. Check network access to modis.ornl.gov")

if failed:
    print(f"\n{len(failed)} Tribal Nations failed:")
    for f in failed:
        print(f"  {f['name']}: {f['error']}")

## Growing Season NDVI

In [ ]:
# Growing season filter and annual means
ndvi_gs = ndvi_raw[ndvi_raw["month"].isin(GROWING_SEASON_MONTHS)].copy()

# Annual growing season mean per Tribal Nation
ndvi_annual = (
    ndvi_gs.groupby(["common_name", "year", "is_primary"])
    .agg(
        ndvi_mean=("ndvi", "mean"),
        ndvi_min=("ndvi",  "min"),
        ndvi_max=("ndvi",  "max"),
        obs_count=("ndvi", "count"),
    )
    .round(4)
    .reset_index()
)

# Condition category
def ndvi_condition(ndvi):
    if pd.isna(ndvi):         return "No data"
    if ndvi < NDVI_POOR:      return "Poor"
    if ndvi < NDVI_FAIR:      return "Fair"
    if ndvi < NDVI_GOOD:      return "Good"
    return                           "Excellent"

ndvi_annual["condition"] = ndvi_annual["ndvi_mean"].apply(ndvi_condition)

# Long-term mean per Tribal Nation
ndvi_summary = (
    ndvi_annual.groupby("common_name")
    .agg(
        mean_growing_ndvi=("ndvi_mean", "mean"),
        min_year_ndvi=("ndvi_mean",  "min"),
        max_year_ndvi=("ndvi_mean",  "max"),
        cv_pct=("ndvi_mean", lambda x: x.std() / x.mean() * 100),
    )
    .round(4)
    .reset_index()
)
ndvi_summary["condition"]  = ndvi_summary["mean_growing_ndvi"].apply(ndvi_condition)
ndvi_summary["is_primary"] = ndvi_summary["common_name"].isin(SD_TRIBES_PRIMARY)

print("GROWING SEASON NDVI SUMMARY BY TRIBAL NATION")
print(f"({NDVI_START_YEAR}–{NDVI_END_YEAR}, months {GROWING_SEASON_MONTHS})")
print(
    ndvi_summary[["common_name", "mean_growing_ndvi", "min_year_ndvi",
                   "cv_pct", "condition"]]
    .sort_values("mean_growing_ndvi", ascending=False)
    .to_string(index=False)
)

## Trend Analysis

In [ ]:
# Linear trend per Tribal Nation (Mann-Kendall via scipy)
# Theil-Sen slope (robust to outliers) and Mann-Kendall p-value
trend_records = []

for name, grp in ndvi_annual.groupby("common_name"):
    grp = grp.sort_values("year").dropna(subset=["ndvi_mean"])
    if len(grp) < 5:
        continue
    years = grp["year"].values.astype(float)
    ndvis = grp["ndvi_mean"].values

    # Theil-Sen slope
    slope, intercept, _, _ = stats.theilslopes(ndvis, years)

    # Mann-Kendall (via scipy linregress p-value as proxy: full MK needs pymannkendall)
    _, _, r, p, _ = stats.linregress(years, ndvis)

    trend_records.append({
        "common_name":       name,
        "theilsen_slope":    round(slope, 6),     # NDVI units per year
        "slope_per_decade":  round(slope * 10, 4),
        "r_squared":         round(r**2, 3),
        "p_value":           round(p, 3),
        "significant":       p < 0.05,
        "direction":         "Increasing" if slope > 0 else "Decreasing",
    })

trend_df = pd.DataFrame(trend_records)
trend_df["is_primary"] = trend_df["common_name"].isin(SD_TRIBES_PRIMARY)

print("GROWING SEASON NDVI TREND THEIL-SEN SLOPE")
print(
    trend_df[["common_name", "slope_per_decade", "direction", "p_value", "significant"]]
    .sort_values("slope_per_decade")
    .to_string(index=False)
)
print("\nNote: slope_per_decade = change in NDVI per 10 years")
print("      Negative = vegetation condition declining over time")

## Drought Year Crosswalk
Cross-reference NDVI low years with the drought events identified in notebook 02.

In [ ]:
# NDVI low years for primary Tribes
# Identify years where growing season NDVI fell below Fair threshold
# These should correspond to severe drought years in notebook 02

primary_annual = ndvi_annual[ndvi_annual["is_primary"]].copy()

low_years = primary_annual[primary_annual["ndvi_mean"] < NDVI_FAIR].copy()
low_years = low_years.sort_values(["common_name", "year"])

print(f"Years with growing season NDVI < {NDVI_FAIR} (Fair threshold):")
print("=" * 55)
for name, grp in low_years.groupby("common_name"):
    yrs = grp["year"].tolist()
    print(f"  {name}: {yrs}")

print()
print("Cross-reference these years with the severe drought years")
print("from notebook 02 (sd_pdsi_monthly.csv) to validate the")
print("NDVI proxy against the PDSI drought record.")

## Visualizations

In [ ]:
# Time series: growing season NDVI for Pine Ridge and Rosebud
CONDITION_COLORS = {
    "Poor":      "#C0392B",
    "Fair":      "#E67E22",
    "Good":      "#27AE60",
    "Excellent": "#1A5276",
}
TRIBE_COLORS = {
    "Oglala Lakota": "#C0392B",
    "Rosebud Sioux": "#1A5276",
}

fig, ax = plt.subplots(figsize=(14, 6))

for name, grp in primary_annual.groupby("common_name"):
    grp = grp.sort_values("year")
    color = TRIBE_COLORS.get(name, "gray")
    ax.plot(
        grp["year"], grp["ndvi_mean"],
        color=color, linewidth=2, alpha=0.9, label=name,
        marker="o", markersize=4,
    )
    # Trend line
    tr = trend_df[trend_df["common_name"] == name]
    if not tr.empty:
        slope = tr["theilsen_slope"].iloc[0]
        years = grp["year"].values
        trend_line = slope * (years - years.mean()) + grp["ndvi_mean"].mean()
        ax.plot(years, trend_line, color=color, linewidth=1.5,
                linestyle="--", alpha=0.6)

# Condition threshold lines
for threshold, label, color in [
    (NDVI_POOR, f"Poor threshold ({NDVI_POOR})",     "#C0392B"),
    (NDVI_FAIR, f"Fair threshold ({NDVI_FAIR})",     "#E67E22"),
    (NDVI_GOOD, f"Good threshold ({NDVI_GOOD})",     "#27AE60"),
]:
    ax.axhline(threshold, color=color, linestyle=":", alpha=0.5,
               linewidth=1.2, label=label)

ax.set_xlabel("Year", fontsize=10)
ax.set_ylabel("Mean Growing Season NDVI", fontsize=10)
ax.set_title(
    "Growing Season NDVI for Pine Ridge and Rosebud\n"
    "Dashed lines = Theil-Sen trend | Dotted = condition thresholds",
    fontsize=11, fontweight="bold",
)
ax.legend(fontsize=9, ncol=2)
ax.set_ylim(0, 0.8)
sns.despine(ax=ax)
plt.tight_layout()

try:
    fig_dir = constants.OUTPUTS_DIR / "figures"
    fig_dir.mkdir(parents=True, exist_ok=True)
    fig.savefig(fig_dir/"03_ndvi_time_series_primary.png", dpi=150, bbox_inches="tight")
except Exception:
    pass
plt.show()

In [ ]:
# All SD Tribes: NDVI time series small multiples
tribes_with_data = ndvi_annual["common_name"].unique()
n      = len(tribes_with_data)
ncols  = 2
nrows  = (n + 1) // ncols

fig, axes = plt.subplots(nrows, ncols, figsize=(13, nrows * 3.5), sharex=True, sharey=True)
axes = np.array(axes).flatten()

for i, name in enumerate(sorted(tribes_with_data)):
    ax   = axes[i]
    grp  = ndvi_annual[ndvi_annual["common_name"] == name].sort_values("year")
    is_p = name in SD_TRIBES_PRIMARY
    color = "#C0392B" if name == "Oglala Lakota" else "#1A5276" if name == "Rosebud Sioux" else "#566573"

    ax.fill_between(grp["year"], 0, grp["ndvi_mean"], alpha=0.2, color=color)
    ax.plot(grp["year"], grp["ndvi_mean"], color=color, linewidth=1.5)
    ax.axhline(NDVI_FAIR, color="#E67E22", linestyle=":", alpha=0.6, linewidth=1)
    ax.set_title(
        name + (" ◄" if is_p else ""),
        fontsize=8, fontweight="bold" if is_p else "normal",
    )
    ax.set_ylim(0.1, 0.75)
    if i % ncols == 0:
        ax.set_ylabel("NDVI", fontsize=8)
    sns.despine(ax=ax)

for ax in axes[n:]:
    ax.set_visible(False)

plt.suptitle(
    "Growing Season NDVI for All South Dakota Tribal Nations\n"
    f"Dotted orange line = Fair threshold ({NDVI_FAIR})  |",
    fontsize=11, fontweight="bold",
)
plt.tight_layout()
try:
    fig.savefig(fig_dir / "03_ndvi_all_tribes.png", dpi=150, bbox_inches="tight")
except Exception:
    pass
plt.show()

In [ ]:
# Trend summary bar chart
fig, ax = plt.subplots(figsize=(10, 6))

td = trend_df.sort_values("slope_per_decade", ascending=True)
colors = [
    "#C0392B" if d == "Decreasing" and s
    else "#E67E22" if d == "Decreasing"
    else "#27AE60" if s
    else "#AED6F1"
    for d, s in zip(td["direction"], td["significant"])
]

bars = ax.barh(td["common_name"], td["slope_per_decade"], color=colors, alpha=0.85)
ax.axvline(0, color="black", linewidth=1)
ax.set_xlabel("NDVI change per decade (growing season mean)", fontsize=10)
ax.set_title(
    "Growing Season NDVI Trend by Tribal Nation\n"
    "Theil-Sen slope; dark = statistically significant (p < 0.05)",
    fontsize=11, fontweight="bold",
)
ax.legend(
    handles=[
        mpatches.Patch(color="#C0392B", label="Significant decrease"),
        mpatches.Patch(color="#E67E22", label="Non-significant decrease"),
        mpatches.Patch(color="#27AE60", label="Significant increase"),
        mpatches.Patch(color="#AED6F1", label="Non-significant increase"),
    ],
    fontsize=8,
)
sns.despine(ax=ax)
plt.tight_layout()
try:
    fig.savefig(fig_dir / "03_ndvi_trend_summary.png", dpi=150, bbox_inches="tight")
except Exception:
    pass
plt.show()

In [ ]:
# Monthly NDVI climatology/growing season phenology
monthly_clim = (
    ndvi_raw[ndvi_raw["is_primary"]]
    .groupby(["common_name", "month"])["ndvi"]
    .agg(["mean", "std"])
    .reset_index()
)
month_labels = ["J","F","M","A","M","J","J","A","S","O","N","D"]

fig, ax = plt.subplots(figsize=(10, 5))
for name, grp in monthly_clim.groupby("common_name"):
    color = TRIBE_COLORS.get(name, "gray")
    ax.plot(grp["month"], grp["mean"], color=color, linewidth=2,
            marker="o", markersize=5, label=name)
    ax.fill_between(
        grp["month"],
        grp["mean"] - grp["std"],
        grp["mean"] + grp["std"],
        color=color, alpha=0.12,
    )
ax.axhline(NDVI_FAIR, color="#E67E22", linestyle=":", alpha=0.6,
           label=f"Fair threshold ({NDVI_FAIR})")
# Shade growing season
ax.axvspan(min(GROWING_SEASON_MONTHS), max(GROWING_SEASON_MONTHS),
           alpha=0.06, color="green", label="Growing season")
ax.set_xticks(range(1, 13))
ax.set_xticklabels(month_labels)
ax.set_ylabel("Mean NDVI ± 1 SD", fontsize=10)
ax.set_title(
    "Monthly NDVI Climatology for Pine Ridge and Rosebud\n"
    f"All years {NDVI_START_YEAR}–{NDVI_END_YEAR}",
    fontsize=11, fontweight="bold",
)
ax.legend(fontsize=9)
ax.set_ylim(0, 0.8)
sns.despine(ax=ax)
plt.tight_layout()
try:
    fig.savefig(fig_dir/"03_ndvi_monthly_climatology.png", dpi=150, bbox_inches="tight")
except Exception:
    pass
plt.show()

## Exports

In [ ]:
try:
    constants.OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)
except FileExistsError:
    pass

ndvi_annual.to_csv(
    constants.OUTPUTS_DIR/"sd_tribal_ndvi_annual.csv", index=False
)
print("Exported to outputs/sd_tribal_ndvi_annual.csv")

ndvi_summary.to_csv(
    constants.OUTPUTS_DIR/"sd_tribal_ndvi_summary.csv", index=False
)
print("Exported to outputs/sd_tribal_ndvi_summary.csv")

trend_df.to_csv(
    constants.OUTPUTS_DIR/"sd_tribal_ndvi_trends.csv", index=False
)
print("Exported to outputs/sd_tribal_ndvi_trends.csv")

## Summary and Findings

*(Fill in after running with your data.)*

**What the data shows:**
- What is the long-term mean growing season NDVI for Pine Ridge and Rosebud?
  Does it fall in the Fair, Good, or Poor condition range?
- Which years had the lowest NDVI on Pine Ridge? Do they correspond to
  the major drought years identified in notebook 02?
- Is there a statistically significant downward trend in growing season
  NDVI on either Pine Ridge or Rosebud? How large is it in NDVI units per decade?
- Does NDVI variability (CV%) differ between Pine Ridge and Rosebud? 
  Does higher variability mean more drought sensitivity?

**Why it matters for agriculture:**
NDVI provides the historical trajectory that Tribal-collected pasture condition
scores can validate and extend. If NDVI shows a multi-year declining trend,
that is a signal that cumulative stress, not just single drought years, is
affecting the land base. That has direct implications for carrying capacity
planning and bison/cattle management.

**What NDVI cannot tell us:**
A low NDVI year might reflect drought, overgrazing, fire, or a shift in
plant community composition toward less palatable species. Distinguishing
between these causes requires on-the-ground observation, which is the
purpose of the Tribal pasture condition monitoring pipeline.

**Connection to the rest of the series:**
- Notebook 06 (System Stress Index) combines NDVI anomaly with PDSI
  to create a compound stress indicator
- Notebook 07 (Climate Projections) shows whether conditions driving
  low NDVI years are projected to become more frequent
- The operational pipeline's pasture condition scoring is the Tribal-led
  ground truth that this satellite analysis can never replace

In [ ]:
# Print citations
print(generate_citations(["census_aiannh", "modis_ndvi"]))